<a href="https://colab.research.google.com/github/kambrave/Modern-AI-Python-Programming/blob/main/AI_video_generaion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q --upgrade --force-reinstall git+https://github.com/huggingface/diffusers.git transformers accelerate torch torchvision safetensors huggingface_hub

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 100.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 95.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 59.0 MB/s eta 0:

# New Section

### ⚠️ IMPORTANT: Restart the Session
Since we updated the packages, you must **restart the Colab runtime** before running the code below:
1. Click **Runtime** -> **Restart session** in the top menu.
2. Run the cells below (starting with the Hugging Face login).

In [ ]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('hanggingFace')
login(token=hf_token)

print("Successfully logged in using Google Colab Secret!")

Successfully logged in using Google Colab Secret!


In [ ]:
import torch
from diffusers import LTXVideoPipeline
from diffusers.utils import export_to_video

# 1. ماڈل کا آئی ڈی (Lightweight model suitable for Colab T4 GPU)
model_id = "Lightricks/LTX-Video"

# 2. পাইپ لائن لوڈ کریں (بجت کے لیے فلوٹ 16 استعمال کر رہے ہیں)
pipe = LTXVideoPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16
).to("cuda")

# اختیاری: میموری بچانے کے لیے
pipe.enable_attention_slicing()

# 3. اپنا پرامپٹ اور نیگیٹو پرامپٹ لکھیں
prompt = "A cinematic shot of a cute orange cat wearing sunglasses riding a skateboard on a beach, sunset lighting, high quality."
negative_prompt = "bad quality, worst quality, deformed, distorted, disfigured, motion blur, extra limbs."

# 4. ویڈیو جنریٹ کریں
print("Generating video... Please wait.")
with torch.autocast("cuda", dtype=torch.bfloat16):
    video_frames = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        height=256,       # کولاب کی میموری کے لحاظ سے ریزولوشن کم رکھی ہے
        width=384,
        num_frames=49,    # فریمز کی تعداد (تقریباً 2 سے 3 سیکنڈ کی ویڈیو)
        fps=24,
        num_inference_steps=30,
        generator=torch.Generator("cuda").manual_seed(42)
    ).frames[0]

# 5. ویڈیو کو محفوظ کریں
output_path = "output_video.mp4"
export_to_video(video_frames, output_path, fps=24)
print(f"Video saved successfully as {output_path}!")

ImportError: cannot import name 'LTXVideoPipeline' from 'diffusers' (/usr/local/lib/python3.13/dist-packages/diffusers/__init__.py)

In [ ]:
from google.colab import files
files.download("output_video.mp4")